# Flood maps for a town: a 2D flood simulation from satellite rain

This notebook makes a flood depth and extent map for one town with MRRpy, from global data only:

1. **Basin** — MRRpy downloads the terrain (FABDEM) around the river basin and finds the basin above an outlet you give.
2. **Storm** — satellite rain (NASA IMERG) for the dates you give, runoff from global curve numbers, roughness from land cover.
3. **Routing** — the river flow over the whole basin, on a 90 m grid.
4. **Flood map** — a 2D flood simulation over a box around the town, on FABDEM at 30 m (or your own LiDAR), with the routed rivers flowing in.

The example is Montpelier, Vermont, in the flood of 10–11 July 2023. On a 32-core workstation the whole notebook runs in a few minutes; on a laptop it takes several times longer.

**You need:** `pip install "MRRpy[gee,notebook]"` and a Google Earth Engine project ([setup](https://mrrpy.readthedocs.io/en/latest/manual/earth-engine/)).

**How big should the box be?** Put it around the place you care about, with the river running through it and its edges across the valley, not along it. Leave a few hundred metres to a kilometre of valley beyond the town, upstream and downstream. 20–100 km² at 30 m runs in seconds to a minute; up to about 500 km² is fine on a workstation.

In [ ]:
import MRRpy
from MRRpy import Config, run_pipeline, plot_inundation, animate_inundation

MRRpy.connect_earth_engine("ee-yourname")      # your Earth Engine project

## 1–4 · Set up and run

The basin is a box for the DEM download and the outlet just below the town; the flood map is a smaller box around the town. Every other setting has a sensible default.

In [ ]:
cfg = Config(
    DEM_PATH="", DEM_SOURCE="fabdem", DEM_SCALE_M=90,
    DEM_BOUNDS_WGS84=(-72.85, 43.95, -72.15, 44.62),     # the Winooski basin (west, south, east, north)
    OUTPUT_POINT=(44.27504, -72.65924),                  # the outlet, just below Montpelier
    GEE_PROJECT="ee-yourname", OUTPUT_DIR="results/montpelier/",
    PRECIP_METHOD="imerg_thiessen",                      # NASA IMERG satellite rain
    EVENT_START_UTC="2023-07-09 00:00", TOTAL_SIMULATION_TIME_HOURS=96,
    RUNOFF_SOURCE="scs_cn",                              # global curve numbers
    MANNINGS_N_SOURCE="lulc",                            # roughness from land cover
    ROUTING_SCHEME="diffusive_implicit",
    INUNDATION_MAP=True,                                 # flood depth and extent maps
    INUNDATION_METHOD="2d",                              # a 2D flood simulation ...
    INUNDATION_AREA=(-72.6527, 44.2201, -72.5122, 44.2847),   # ... over this box around the town
)
out = run_pipeline(cfg)

Prefer to click? `MRRpy.ConfigForm()` opens the same settings as a form. In its map, the rectangle tool sets either the **area to download** or the **area for the flood maps**: choose which above the map, then draw.

In [ ]:
# form = MRRpy.ConfigForm(cfg)   # the same settings, as a form with a map
# form

## Look at the flood

The deepest water each place reached, the flood at any hour, and an animation with the hydrograph.

In [ ]:
plot_inundation(out)                    # deepest water over the run, outlined
plot_inundation(out, time=60.0)         # the flood 60 hours into the run
animate_inundation(out, out_path="results/montpelier/flood.gif")

## Check the run

`inundation_summary.json` says how long the simulation took, whether any water was lost or created (the water balance error should be about 10⁻¹⁴), and whether the flood reached an edge of the box. If it did, the edge held the water back like a wall: make the box larger there and redraw. `inflows.csv` lists the rivers flowing into the box and their peak flows.

In [ ]:
import json, pandas as pd

summary = json.load(open(out["summary_json"]))
for key in ("flooded_km2", "max_depth_m", "max_speed_m_s", "simulated_hours",
            "water_balance_error", "seconds", "water_against_edge"):
    print(f"{key:22s} {summary[key]}")
pd.read_csv(out["inflows_csv"])

## Another box, another DEM, the other method: no routing again

The routed flows are saved, so the flood maps can be redrawn in seconds. With your own LiDAR, use `INUNDATION_DEM="file"` and `INUNDATION_DEM_PATH`.

In [ ]:
cfg.INUNDATION_AREA = (-72.62, 44.24, -72.57, 44.27)        # just downtown
# cfg.INUNDATION_DEM, cfg.INUNDATION_DEM_PATH = "file", "lidar_1m.tif"
# cfg.INUNDATION_METHOD = "hand"                              # quick maps for every river
out2 = run_pipeline(cfg, stages=["inundation"])
plot_inundation(out2)